# Analisis del agente Rafa: TBOPI con memoria global

Este notebook estudia el aporte de guardar Q-values globales en el agente Rafa. La comparacion principal es una ablation local:

- **RafaBase**: version base. Root UCB con rollouts aleatorios, sin Q-table global.
- **RafaQ**: misma base, pero usando la Q-table entrenada por self-play como prior.
- **RafaImproved / Rafa**: version final. Usa Q-table global, rollouts heuristicos y seleccion final robusta.

La memoria global se obtuvo con una sesion de entrenamiento por **self-play de 50,000 partidas**, usando **24 inner rollouts por estado** en el subproceso local UCB. Durante ese entrenamiento, los inner trials solo se usan para mejorar la generacion del outer trial; la Q-table global se actualiza con el resultado final de la partida completa.

La version final incluye tres mejoras de TBOPI sobre el baseline inicial: rollouts heurísticos, seleccion final robusta por visitas/valor, y mayor peso del prior global (`global_prior_visits=10`).

La pregunta experimental es: **que cambia al pasar de una politica base sin memoria, a una con Q-values globales, y finalmente a una version TBOPI mejorada?**

## 0. Metodologia del entrenamiento

El agente final se entreno offline con self-play:

- Numero de partidas de entrenamiento: **50,000**.
- Generador de partidas: Rafa contra una copia de si mismo.
- En cada estado del outer trial se ejecuto un subproceso local con UCB.
- Presupuesto del subproceso local: **24 inner rollouts** por estado.
- Politica de rollout: heuristica tactica, no puramente aleatoria. Primero intenta ganar, luego bloquear, luego evitar entregar victoria inmediata, y finalmente prefiere columnas centrales con una pequena exploracion.
- Seleccion final: accion robusta con mas visitas locales y, como desempate, mayor valor estimado.
- Peso de memoria global en UCB local: `global_prior_visits=10` por defecto.
- Recompensa final: `+1` victoria, `0` empate, `-1` derrota.
- La Q-table global se guarda en `rafa_q_values.pkl` y luego se usa como prior en la politica online.

Esto mantiene la idea de **trial-based online policy improvement**: durante la generacion de cada partida se mejora localmente la decision con inner trials, y luego la experiencia del outer trial completo alimenta la memoria global.

## 0.1. Versiones evaluadas

Para que la comparacion no sea atomica, se separan tres versiones del agente:

| Version | Q-table global | Rollout | Seleccion final |
|---|---:|---|---|
| RafaBase | No | Aleatorio | Proporcional a visitas |
| RafaQ | Si (`global_prior_visits=5`) | Aleatorio | Proporcional a visitas |
| RafaImproved / Rafa | Si (`global_prior_visits=10`) | Heuristico | Robusta: visitas y valor |

La motivacion del self-play es que no se asume conocer la politica optima. En vez de entrenar contra un experto, el agente genera experiencia contra copias de si mismo y mejora su memoria global a partir de esas trayectorias.

## 1. Setup

Las celdas leen los resultados guardados en `groups/Rafa/analytics/`. Si quieres regenerar resultados, usa el script:

```powershell
python groups\Rafa\compare_tournament_agents.py --target Rafa --opponents RafaNoQ --games-per-match 25 --total-time 60 --output-dir groups\Rafa\analytics\tournament_style
```

In [ ]:
from pathlib import Path
import csv
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "connect4").exists() and (path / "groups").exists():
            return path
    raise FileNotFoundError("No pude encontrar la raiz del repo Connect4RL.")


ROOT = find_repo_root(Path.cwd().resolve())
RAFA_DIR = ROOT / "groups" / "Rafa"
ANALYTICS = RAFA_DIR / "analytics"
TOURNAMENT = ANALYTICS / "tournament_style"

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

print("Root:", ROOT)
print("Analytics:", ANALYTICS)

In [ ]:
def load_games_csv(path: Path) -> pd.DataFrame:
    """Carga games.csv incluso si mezcla filas antiguas y nuevas con run_started_at."""
    rows = []
    if not path.exists():
        return pd.DataFrame()

    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        for raw in reader:
            if not raw:
                continue
            if raw == header:
                continue

            if len(raw) == len(header):
                row = dict(zip(header, raw))
                row.setdefault("run_started_at", "")
            elif len(raw) == len(header) + 1:
                row = {"run_started_at": raw[0]}
                row.update(dict(zip(header, raw[1:])))
            else:
                continue
            rows.append(row)

    df = pd.DataFrame(rows)
    numeric_cols = [
        "game", "winner_color", "moves_count", "red_time", "yellow_time"
    ]
    for col in numeric_cols:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


games = load_games_csv(TOURNAMENT / "games.csv")
print("Partidas cargadas:", len(games))
games.tail()

## 2. Experimento principal: evolucion del agente

El experimento ideal para la entrega compara `RafaBase`, `RafaQ` y `RafaImproved`. Si estos datos aun no existen, se pueden generar con:

```powershell
python groups\Rafa\compare_tournament_agents.py --include RafaBase RafaQ RafaImproved --games-per-match 10 --total-time 60 --output-dir groups\Rafa\analytics\version_ablation
```

La comparacion `Rafa` vs `RafaNoQ` queda como ablation complementaria de memoria global bajo la version online actual.

In [ ]:
version_games_path = ANALYTICS / "version_ablation" / "games.csv"
version_games = load_games_csv(version_games_path)
versions = ["RafaBase", "RafaQ", "RafaImproved"]

if version_games.empty:
    print("No encontre resultados de version_ablation en", version_games_path)
    print("Genera el experimento con el comando mostrado arriba.")
else:
    version_rows = []
    for version in versions:
        played = version_games[(version_games["red"] == version) | (version_games["yellow"] == version)]
        wins = int((played["winner"] == version).sum())
        draws = int((played["winner"] == "draw").sum())
        losses = int(len(played) - wins - draws)
        version_rows.append({"version": version, "wins": wins, "losses": losses, "draws": draws, "games": len(played)})
    version_summary = pd.DataFrame(version_rows)
    version_summary["win_rate"] = version_summary["wins"] / version_summary["games"].replace(0, np.nan)
    display(version_summary)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(version_summary["version"], version_summary["win_rate"], color=["#8c8c8c", "#345995", "#2f6f4e"])
    ax.set_ylim(0, 1.02)
    ax.set_title("Evolucion del agente: base -> Q -> mejorado")
    ax.set_ylabel("Win rate en ablation interna")
    for i, row in version_summary.iterrows():
        ax.text(i, row["win_rate"] + 0.02, f"{row['win_rate']:.2f}", ha="center")
    plt.tight_layout()
    plt.show()

## 3. Ablation complementaria: Rafa actual vs RafaNoQ

Este experimento compara dos versiones del mismo agente bajo las mismas reglas. La unica diferencia conceptual es si se usa o no la Q-table global como prior del UCB local.

In [ ]:
def filter_rafa_vs_noq(df: pd.DataFrame) -> pd.DataFrame:
    expected_columns = [
        "run_started_at", "game", "match", "red", "yellow", "winner",
        "winner_color", "moves_count", "red_time", "yellow_time",
        "red_over_time", "yellow_over_time", "invalid_by",
        "rafa_color", "result_for_rafa",
    ]
    if df.empty or not {"red", "yellow", "winner"}.issubset(df.columns):
        return pd.DataFrame(columns=expected_columns)
    mask = (
        ((df["red"] == "Rafa") & (df["yellow"] == "RafaNoQ")) |
        ((df["red"] == "RafaNoQ") & (df["yellow"] == "Rafa"))
    )
    out = df.loc[mask].copy()
    out["rafa_color"] = np.where(out["red"] == "Rafa", "red", "yellow")
    out["result_for_rafa"] = np.select(
        [out["winner"] == "Rafa", out["winner"] == "RafaNoQ", out["winner"] == "draw"],
        ["win", "loss", "draw"],
        default="other",
    )
    return out


rvn = filter_rafa_vs_noq(games)
if rvn.empty:
    print("No encontre partidas Rafa vs RafaNoQ en", TOURNAMENT / "games.csv")
    print("Regenera datos con la celda opcional del final o con compare_tournament_agents.py.")

summary = rvn["result_for_rafa"].value_counts().reindex(["win", "loss", "draw"], fill_value=0)
display(summary.to_frame("games"))
print("Total Rafa vs RafaNoQ:", int(summary.sum()))

In [ ]:
fig, ax = plt.subplots()
colors = ["#2f6f4e", "#b54c3f", "#8c8c8c"]
summary.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Rafa actual vs RafaNoQ")
ax.set_xlabel("Resultado para Rafa")
ax.set_ylabel("Numero de partidas")
ax.tick_params(axis="x", rotation=0)
for idx, value in enumerate(summary.values):
    ax.text(idx, value + 0.2, str(int(value)), ha="center")
plt.tight_layout()
plt.show()

## 3. Desempeno por color

El reto pide mirar el desempeno considerando ambos colores. En Connect-4, mover primero importa, por eso se separan los resultados cuando Rafa juega como rojo y como amarillo.

In [ ]:
by_color = (
    rvn.groupby(["rafa_color", "result_for_rafa"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["win", "loss", "draw"], fill_value=0)
)
display(by_color)

fig, ax = plt.subplots()
by_color.plot(kind="bar", stacked=False, ax=ax, color=colors)
ax.set_title("Resultado de Rafa vs RafaNoQ por color")
ax.set_xlabel("Color de Rafa")
ax.set_ylabel("Numero de partidas")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 4. Uso de recursos: tiempo y longitud de partidas

Ambas versiones usan UCB local. La version con memoria no reemplaza la busqueda online; la inicializa con informacion aprendida. Por eso es util mirar si gana con partidas mas cortas/largas y cuanto tiempo consume.

In [ ]:
rvn = rvn.copy()
rvn["rafa_time"] = np.where(rvn["red"] == "Rafa", rvn["red_time"], rvn["yellow_time"])
rvn["noq_time"] = np.where(rvn["red"] == "RafaNoQ", rvn["red_time"], rvn["yellow_time"])

resource_summary = rvn[["moves_count", "rafa_time", "noq_time"]].agg(["mean", "median", "min", "max"]).round(2)
display(resource_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
rvn.boxplot(column="moves_count", by="result_for_rafa", ax=axes[0])
axes[0].set_title("Movimientos por resultado")
axes[0].set_xlabel("Resultado para Rafa")
axes[0].set_ylabel("Movimientos")

axes[1].boxplot([rvn["rafa_time"].dropna(), rvn["noq_time"].dropna()], labels=["Rafa", "RafaNoQ"])
axes[1].set_title("Tiempo usado por partida")
axes[1].set_ylabel("Segundos")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 5. Tamano de la memoria aprendida

La Q-table representa la memoria global aprendida por self-play. En la politica actual, estos valores se usan como prior de UCB: no sustituyen los inner trials, pero sesgan el proceso local hacia acciones que funcionaron antes.

In [ ]:
qtable_path = RAFA_DIR / "rafa_q_values.pkl"
if qtable_path.exists():
    with qtable_path.open("rb") as f:
        payload = pickle.load(f)
    q_table = payload.get("q_table", payload) if isinstance(payload, dict) else payload
    q_states = len(q_table)
    q_entries = sum(len(actions) for actions in q_table.values())
    print(f"Estados con memoria: {q_states:,}")
    print(f"Pares (estado, accion): {q_entries:,}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Estados", "Pares (s,a)"], [q_states, q_entries], color=["#345995", "#03cea4"])
    ax.set_title("Tamano de la Q-table global")
    ax.set_ylabel("Conteo")
    ax.ticklabel_format(style="plain", axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("No se encontro rafa_q_values.pkl. Ejecuta train_self_play.py para generar memoria.")

## 6. Sweep de parametros: memoria vs presupuesto online

Para profundizar el analisis, variamos dos parametros numericos:

- `total_time`: presupuesto de computo por agente/partida.
- `global_prior_visits`: peso maximo con el que la Q-table entra como prior del UCB local. En la version final usamos `10` como default.

Esto permite responder si la memoria global ayuda mas cuando hay poco tiempo para pensar online.

In [ ]:
sweep_path = ANALYTICS / "memory_sweep" / "memory_sweep.csv"
if sweep_path.exists():
    sweep = pd.read_csv(sweep_path)
else:
    sweep = pd.DataFrame()
    print("No encontre", sweep_path)
    print("Genera datos con la celda opcional de sweep o con sweep_memory_ablation.py")

display(sweep)

In [ ]:
if not sweep.empty:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for prior, data in sweep.groupby("global_prior_visits"):
        data = data.sort_values("total_time")
        ax.plot(data["total_time"], data["memory_win_rate"], marker="o", label=f"prior={prior}")
    ax.set_xscale("log")
    ax.set_xlabel("Tiempo por agente/partida (s)")
    ax.set_ylabel("Win rate de Rafa vs RafaNoQ")
    ax.set_title("Aporte de la memoria segun presupuesto online")
    ax.set_ylim(0, 1.02)
    ax.legend(title="global_prior_visits")
    plt.tight_layout()
    plt.show()

    pivot = sweep.pivot(index="global_prior_visits", columns="total_time", values="memory_win_rate")
    fig, ax = plt.subplots(figsize=(8, 4.8))
    im = ax.imshow(pivot.values, vmin=0, vmax=1, cmap="YlGn", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)), [str(c) for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)), [str(i) for i in pivot.index])
    ax.set_xlabel("Tiempo por agente/partida (s)")
    ax.set_ylabel("global_prior_visits")
    ax.set_title("Heatmap: win rate de Rafa con memoria")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center")
    fig.colorbar(im, ax=ax, label="Win rate")
    plt.tight_layout()
    plt.show()

### Celda opcional: generar el sweep

Este experimento puede tardar. Para una corrida rapida usa pocos juegos; para resultados presentables sube `--games-per-side`.

In [ ]:
# import subprocess, sys
# cmd = [
#     sys.executable,
#     "groups/Rafa/sweep_memory_ablation.py",
#     "--times", "0.1", "0.3", "1", "2",
#     "--prior-visits", "1", "5", "10", "20",
#     "--games-per-side", "20",
# ]
# subprocess.run(cmd, check=True)

## 7. Conclusiones preliminares

- RafaNoQ es un baseline fuerte porque conserva las heuristicas tacticas y UCB local.
- Rafa actual agrega memoria global: la Q-table entra como prior de los inner trials.
- Si la diferencia no es enorme, eso no significa que la memoria sea inutil: significa que la busqueda online ya explica una parte grande del desempeno.
- La hipotesis mas defendible es que la memoria global ayuda especialmente cuando el presupuesto online es limitado o cuando el estado ya fue visto muchas veces durante self-play.
- La principal debilidad frente a agentes tipo MCTS es que Rafa no construye un arbol profundo durante el turno; hace mejora local desde la raiz.

## 8. Celda opcional: regenerar experimento base

Descomenta y ejecuta esta celda si quieres generar nuevos resultados. Por defecto esta desactivada para que el notebook sea rapido de abrir.

In [ ]:
# import subprocess, sys
# cmd = [
#     sys.executable,
#     "groups/Rafa/compare_tournament_agents.py",
#     "--target", "Rafa",
#     "--opponents", "RafaNoQ",
#     "--games-per-match", "25",
#     "--total-time", "60",
#     "--output-dir", "groups/Rafa/analytics/tournament_style",
# ]
# subprocess.run(cmd, check=True)